In [0]:
# %pip install openpyxl

In [0]:
%restart_python

In [0]:
import openpyxl
print(openpyxl.__version__)

inport Employees Table

In [0]:
from openpyxl import load_workbook

file_path= "/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)

sheet=workbook["employees"]

data=list(sheet.values)

headers=data[0]

rows=data[1:]

print(headers)
print("Number of rows:",len(rows))

employees=spark.createDataFrame(rows,schema=headers)
display(employees)


import Employees_service table

In [0]:
from openpyxl import load_workbook

file_path="/Volumes/main/training/hr_data/HR_Dataset.xlsx"

workbook=load_workbook(file_path,data_only=True)
sheet=workbook["employee_service"]
data=list(sheet.values)

headers=data[0]

rows=data[1:]
print(headers)
print("Number of rows",len(rows))

employee_service=spark.createDataFrame(rows,schema=headers)
display(employee_service)

In [0]:
orders=spark.read.csv("/Volumes/main/training/csv/orders.csv",header=True,inferSchema=True)
display(orders)

Question 1. Input Tables / Columns: employee_service (emp_id, hire_date, resignation_date)
Task Requirements:
•	Calculate total calendar days between hire date and resignation date.
•	For active employees whose resignation date is missing, measure tenure dynamically up to today's date.
•	Display employee identifier, hire date, resignation date, and computed service days.
•	Sort workforce records from longest tenure to shortest tenure.
Expected Output Specification: DataFrame with columns [emp_id, hire_date, resignation_date, total_service_days], ordered by service days descending.
Guiding Hints & Formulas: Formula / Concept: Service Days = Coalesce(Resignation Date, Today) - Hire Date. Investigate PySpark date difference and null fallback functions.


In [0]:
from pyspark.sql.functions import col,coalesce,current_date,datediff

result=( 
    employee_service
    .withColumn(
        "total_service_days",
        datediff(
            coalesce(col("resignation_date"),current_date()),
            col("hire_date")
        )
).select(
    "emp_id",
    "hire_date",
    "resignation_date",
    "total_service_days"
    
    ).orderBy(col("total_service_days").desc())
)

display(result)

Question 2. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Decompose each employee's hire date into its constituent calendar components: calendar year, month number, day of month, day of week, day of year, and fiscal quarter.
•	Ensure day of week indicates position within the standard week.
•	Display employee identifier and hire date alongside all extracted calendar fields.
Expected Output Specification: DataFrame with 8 columns including employee identifier, hire date, and all individual calendar components.
Guiding Hints & Formulas: Formula / Concept: Explore PySpark temporal extraction functions for year, month, day, weekday, day-of-year, and quarter.


In [0]:
from pyspark.sql.functions import year,day,month,quarter,dayofyear,dayofweek,dayofmonth


result=employees.select(
    "employee_id",
    "hire_date",
    year(col("hire_date")).alias("Year"),
    month(col("hire_date")).alias("Month"),
    dayofmonth(col("hire_date")).alias("Day"),
    dayofweek(col("hire_date")).alias("Day Week"),
    dayofyear(col("hire_date")).alias("Day year"),
    quarter(col("hire_date")).alias("Quarter")
)

display(result)

Question 3. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Convert hire date into standard US numerical format (Month/Day/Year).
•	Convert hire date into formal corporate text format (Abbreviated Month Name Day, Full Year).
•	Convert hire date into an executive long-form presentation string (Full Day Name, Full Month Name Day, Full Year).
Expected Output Specification: DataFrame with columns [employee_id, hire_date, us_format, formal_format, full_text_format].
Guiding Hints & Formulas: Formula / Concept: Explore date formatting pattern strings in PySpark (e.g. 'MM/dd/yyyy', 'MMM dd, yyyy', 'EEEE, MMMM dd, yyyy').


In [0]:
from pyspark.sql.functions import date_format,col

result=employees.select(
    "employee_id",
    "hire_date",
    date_format("hire_date","MM/dd/yyyy").alias("us_format"),
    date_format("hire_date","MMM dd,yyyy").alias("formal_format"),
    date_format("hire_date","EEEE, MMMM dd, yyyy").alias("full_text_format")
)

display(result)

Question 4.Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Project the 90-calendar-day initial probation end date from the employee's hire date.
•	Project the 180-calendar-day mid-year performance review milestone.
•	Calculate an early-warning deadline exactly 14 calendar days prior to the probation end date for HR audit.
Expected Output Specification: DataFrame with columns [employee_id, hire_date, probation_end_date, mid_year_review_date, probation_warning_deadline].
Guiding Hints & Formulas: Formula / Concept: Milestone Date = Base Date + Number of Days. Warning Date = Milestone Date - Offset Days.


In [0]:
from pyspark.sql.functions import col,date_add

result=employees.withColumn(
    "probation_end_date",
    date_add("hire_date",90)
).withColumn(
    "mid_year_review_date",
    date_add("hire_date",180) 
).withColumn(
    "probation_warning_deadline",
    date_add("probation_end_date",-14)
).select(
    "employee_id",
    "hire_date",
    "probation_end_date",
    "mid_year_review_date",
    "probation_warning_deadline"
)

display(result)



Question 5. **Input Tables / Columns:** employees (employee_id, hire_date)

**Task Requirements:**

·       Determine the calendar start date (first day of the month) corresponding to each employee's hire date.

·       Determine the calendar end date (last day of the month) for that same month.

·       Calculate the total calendar days contained in that specific hiring month.

·       Calculate how many calendar days remained in the hiring month from the employee's start date onwards.

**Expected Output Specification:** DataFrame with columns [employee_id, hire_date, month_start_date, month_end_date, total_month_days, remaining_hire_days].

**Guiding Hints & Formulas:** *Formula / Concept: Total Month Days = (Last Day of Month - First Day of Month) + 1. Remaining Days = (Last Day - Hire Date) + 1.*    

In [0]:
from pyspark.sql.functions import col,trunc,last_day,datediff,day


result=employees.withColumn(
    "month_start_date",
    trunc("hire_date","month")
).withColumn(
    "month_end_date",
    last_day("hire_date")
).select(
    "employee_id",
    "hire_date",
    "month_start_date",
    "month_end_date",
    #option 1 use datediff to find the number of days in the month
    # (datediff(
    #         col("month_end_date"),
    #         col("month_start_date")
    #     )+1
    #).alias("total_month_days")

    # option 2 i use last_day because last day is the end #date of month so both count is same if the month have 30 #days the last date is 30 so i use this logic to find the #last day

    day(last_day(col("hire_date"))
    ).alias("total_month_days"),
    (datediff(
            col("month_end_date"),
            col("hire_date")
        )+1
    ).alias("remaining_hire_days"),
)

display(result)

Question 6. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Calculate each employee's 1-year corporate anniversary date.
•	For each employee's hire date, locate the exact calendar date of the immediate following Friday for weekly onboarding.
•	Display employee identifier, hire date, anniversary date, and next Friday orientation date.
Expected Output Specification: DataFrame with columns [employee_id, hire_date, anniversary_date, next_orientation_friday].
Guiding Hints & Formulas: Formula / Concept: Anniversary Date = Base Date + 12 Months. Explore functions that advance dates to the next named weekday (e.g. 'next_day').


In [0]:
from pyspark.sql.functions import col,add_months,next_day

result=employees.withColumn(
    "Anniversary_Date",
    add_months(col("hire_date"),12)
)

result=result.withColumn(
    "next_orientation_friday",
    next_day(col("hire_date"), "Fri")
)

ans=result.select(
    "employee_id",
    "hire_date",
    "Anniversary_Date",
    "next_orientation_friday"
)

display(ans)

Question 7. Input Tables / Columns: orders (OrderID, OrderDate, ShippedDate, RequiredDate)
Task Requirements:
•	Calculate fulfillment turnaround duration: calendar days elapsed between order placement and warehouse shipment.
•	Calculate customer tolerance buffer: calendar days between shipment date and required delivery date.
•	Identify orders that experienced delayed fulfillment where shipment occurred after the required delivery date.
Expected Output Specification: DataFrame with columns [OrderID, OrderDate, ShippedDate, RequiredDate, fulfillment_days, delivery_buffer_days, is_late_shipment].
Guiding Hints & Formulas: Formula / Concept: Fulfillment Days = Shipped Date - Order Date. Late Flag = Shipped Date > Required Date.


In [0]:
from pyspark.sql.functions import col,when,date_diff

result=orders.withColumn(
    "fulfillment_days",
    #option 1 you can use the subtraction you get the output like INTERVAL '2' DAYS

    # col("ShippedDate")-col("OrderDate")

    #option 2 use date diff
    date_diff(col("ShippedDate"),col("OrderDate"))
).withColumn(
    "Delivery_Buffer",
    # col("RequiredDate")-col("ShippedDate")
    date_diff(col("RequiredDate"),col("ShippedDate"))
)

ans=result.select(
    "OrderID",
    "OrderDate",
    "ShippedDate",
    "RequiredDate",
    "fulfillment_days",
    "Delivery_Buffer",
    when(
        col("ShippedDate")>col("RequiredDate"),
        True
    ).otherwise(
        False
    ).alias("is_late_shipment")
)

display(ans)

Question 8. Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Calculate exact fractional months elapsed between hire date and today's date.
•	Calculate rounded integer months of completed service.
•	Compute total completed full calendar years of service.
•	Sort staff by total completed service months descending.
Expected Output Specification: DataFrame with columns [employee_id, hire_date, fractional_service_months, completed_service_months, service_years].
Guiding Hints & Formulas: Formula / Concept: Investigate PySpark functions designed to calculate elapsed months between two temporal endpoints (accounting for varying month lengths).


In [0]:
from pyspark.sql.functions import col,months_between,current_date,round,floor

result=employees.withColumn(
    "fractional_service_months",
    months_between(current_date(),col("hire_date"))
).withColumn(
    "completed_service_months",
    round(
        col("fractional_service_months"),0
    )
).withColumn(
    "service_year",
    floor(col("completed_service_months")/12)
)

ans=result.select(
    "employee_id",
    "hire_date",
    "fractional_service_months",
    "completed_service_months",
    "service_year"
).orderBy(col("completed_service_months").desc())
display(ans)

Question 9. 
Input Tables / Columns: employees (employee_id, hire_date)
Task Requirements:
•	Calculate completed years of service for each employee as of the current date.
•	Categorize staff into seniority bands:
•	- 20+ years: 'VETERAN (20+ YRS)'
•	- 15 to 19 years: 'SENIOR (15-19 YRS)'
•	- 10 to 14 years: 'INTERMEDIATE (10-14 YRS)'
•	- 5 to 9 years: 'EXPERIENCED (5-9 YRS)'
•	- Under 5 years: 'ASSOCIATE (< 5 YRS)'
•	Count total staff allocated to each seniority band.
Expected Output Specification: Summary DataFrame with columns [seniority_band, headcount] sorted by seniority tier.
Guiding Hints & Formulas: Formula / Concept: Service Years = Floor(Months Elapsed / 12). Apply multi-condition branching to assign seniority bands.


In [0]:
from pyspark.sql.functions import col,months_between,current_date,floor,when

result=employees.withColumn(
    "service_year",
    floor(
        months_between(
            current_date(),col("hire_date")
        )/12
    )
)

ans=result.select(
    "employee_id",
    "hire_date",
    "service_year",
    when(
        col("service_year")>=20,
        "VETERAN"
    ).when(
        (col("service_year")>=15) & (col("service_year")<=19),
        "SENIOR"
    ).when(
        (col("service_year")>=10) & (col("service_year")<=14),
        "INTERMEDIATE"
    ).when(
        (col("service_year")>=5) & (col("service_year")<=9),
        "EXPERIENCED"
    ).otherwise(
        "ASSOCIATE"
    ).alias("Band")
)

summary=ans.groupBy("Band").count().withColumnRenamed("count","headcount")

summary=summary.withColumn(
    "tier_order",
    when(
        col("Band")=="VETERAN",
        1
    ).when(
        col("Band")=="SENIOR",
        2
    ).when(
        col("Band")=="INTERMEDIATE",
        3
    ).when(
        col("Band")=="EXPERIENCED",
        4
    ).otherwise(
        5
    )
)

summary=summary.orderBy(
    "tier_order"
).drop("tier_order")

display(summary)




Question 10. Input Tables / Columns: employees (employee_id, hire_date, salary)
Task Requirements:
•	Truncate each employee's hire date to the starting day of its financial quarter.
•	Compute total personnel hired and average starting compensation within each financial quarter.
•	Identify the calendar year and quarter label (e.g., '2005-Q1') for each cohort.
•	Order cohorts chronologically.
Expected Output Specification: DataFrame with columns [quarter_start_date, quarter_label, hires_count, avg_salary] ordered chronologically.
Guiding Hints & Formulas: Formula / Concept: Investigate date truncation options (such as truncating by quarter or month) to normalize timestamps to cohort start dates.


In [0]:
from pyspark.sql.functions import col,when,trunc,avg,count,round,concat,year,quarter,lit


result=employees.withColumn(
    "quarter_starting_date",
    trunc(col("hire_date"),"quarter")
)

result=result.withColumn(
    "quarter_label",
    concat(
        year(col("hire_date")),
        lit("Q"),
        quarter(col("hire_date")
    )
    )
)
result=result.groupBy(
    "quarter_starting_date",
    "quarter_label"
).agg(
    count("*").alias("count"),
    round(avg("salary"),2).alias("avg_salary")
).orderBy(
    col("quarter_starting_date")
)
display(result)

Question 11. Input Tables / Columns: orders (OrderID, OrderDate, CustomerID)
Task Requirements:
•	Identify the single most recent order date present in the dataset (anchor date).
•	Filter orders placed within the last 90 calendar days relative to the anchor date.
•	Filter orders placed within the last 365 calendar days relative to the anchor date.
•	Compute order count and distinct customer count for each lookback window.
Expected Output Specification: Summary DataFrame comparing 90-day vs 365-day rolling order counts and customer counts.
Guiding Hints & Formulas: Formula / Concept: Lookback Window Condition = Order Date >= Date Subtract(Anchor Date, Window Days).


In [0]:
from pyspark.sql.functions import col,date_sub,max,count,countDistinct

# find the most recent order date
anchor_date=orders.agg(
    max("OrderDate").alias("Anchor_date")
).collect()[0]["Anchor_date"]

# 90-days window

orders_90 = orders.filter(
    col("OrderDate")>= date_sub(lit(anchor_date),90)
)

#365-days window

orders_365 = orders.filter(
    col("OrderDate")>= date_sub(lit(anchor_date),365)
)

ans_90 = orders_90.agg(
    count("*").alias("order_count"),
    countDistinct("CustomerID").alias("distinct_customer_count")
).withColumn(
    "lookback_window",
    lit("90 Days")
)


ans_365 = orders_365.agg(
    count("*").alias("order_count"),
    countDistinct("CustomerID").alias("distinct_customer_count")
).withColumn(
    "lookback_window",
    lit("365 Days")
)

ans= ans_90.unionByName(ans_365).select(
    "lookback_window",
    "order_count",
    "distinct_customer_count"
)

display(ans)

Question 12. 
Input Tables / Columns: employee_service (emp_id, hire_date, resignation_date)
Task Requirements:
•	Filter to former employees having a recorded resignation date.
•	Calculate exact service tenure in days, months, and years at departure.
•	Determine whether the employee resigned during their first year of employment (First-Year Flight Risk).
•	Calculate average tenure in years among all resigned employees.
Expected Output Specification: DataFrame with columns [emp_id, hire_date, resignation_date, service_days, service_years, is_first_year_departure].
Guiding Hints & Formulas: Formula / Concept: First-Year Departure = Service Days <= 365. Average Tenure = Mean(Service Years).


In [0]:
from pyspark.sql.functions import col,floor,datediff,months_between

result=employee_service.filter(
    col("resignation_date").isNotNull()
)
result=result.withColumn(
    "service_days",
    datediff(
        col("resignation_date"),col("hire_date")
    )
).withColumn(
    "service_year",
    floor(col("service_days")/365)
).withColumn(
    "service_month",
    round(
        months_between(col("resignation_date"),col("hire_date")),2)
)

ans=result.select(
    "emp_id",
    "hire_date",
    "resignation_date",
    "service_days",
    "service_year",
    "service_month",
    when(
        col("service_days")<=365,
        True
    ).otherwise(False).alias("is_first_year_departure"),
)

# display(result)

avg_tenure=result.agg(
    round(
        avg("service_year"),2
    ).alias("average_tenure_years")
)

display(avg_tenure)